# KINNEWS – GPU experiments (Colab / Kaggle)

Runs the neural experiments: BiLSTM (E7–E9) and Transformer fine-tuning (T1–T4).
**Runtime → Change runtime type → T4 GPU** before starting. Total time ≈ 45–60 min.

At the end, `results.zip` is downloaded — unzip it into the repo root locally so the
results sit next to the baseline results.

In [2]:
REPO = "https://github.com/Samkwizera/NLP-application.git"   # <-- your repo URL
!git clone -q $REPO repo
%cd repo
!pip install -q -r requirements-train.txt
!python scripts/prepare_data.py
import torch; print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE - switch runtime!")

/kaggle/working/repo
{
  "official_train_rows": 17014,
  "official_test_rows": 4254,
  "train_conflicting_label_articles_dropped": 160,
  "train_test_overlap_dropped": 3,
  "train_rows": 6344,
  "val_rows": 1120,
  "test_rows": 3464,
  "train_original_rows": 17014
}
GPU: Tesla T4


## BiLSTM experiments (course: embeddings, RNN/LSTM, attention)
- **E7** random embeddings, max pooling
- **E8** Word2Vec-initialised embeddings, max pooling → does unsupervised pretraining on the same small corpus help?
- **E9** Word2Vec + additive attention pooling → does attention help on long articles?

In [3]:
!python scripts/train_bilstm.py --run-name e7_bilstm_random_max --pooling max
!python scripts/train_bilstm.py --run-name e8_bilstm_w2v_max --pooling max --w2v
!python scripts/train_bilstm.py --run-name e9_bilstm_w2v_attention --pooling attention --w2v

epoch  1  train loss 2.1179  val macro-F1 0.1394
epoch  2  train loss 1.7458  val macro-F1 0.2867
epoch  3  train loss 1.3629  val macro-F1 0.3492
epoch  4  train loss 1.1156  val macro-F1 0.3535
epoch  5  train loss 0.9663  val macro-F1 0.3789
epoch  6  train loss 0.8747  val macro-F1 0.3956
epoch  7  train loss 0.7855  val macro-F1 0.4238
epoch  8  train loss 0.7062  val macro-F1 0.4504
epoch  9  train loss 0.6504  val macro-F1 0.4662
epoch 10  train loss 0.5816  val macro-F1 0.4813
epoch 11  train loss 0.5221  val macro-F1 0.4818
epoch 12  train loss 0.4853  val macro-F1 0.4930
epoch 13  train loss 0.4323  val macro-F1 0.4640
epoch 14  train loss 0.3915  val macro-F1 0.5090
epoch 15  train loss 0.3454  val macro-F1 0.5056
e7_bilstm_random_max                     val macro-F1 0.5090 | test acc 0.6094 macro-F1 0.3833
Word2Vec coverage: 50000/50000 vocabulary words
epoch  1  train loss 1.5848  val macro-F1 0.3181
epoch  2  train loss 1.0339  val macro-F1 0.4035
epoch  3  train loss 0.7

## Transformer fine-tuning
Research question: does pretraining that *includes Kinyarwanda* beat general multilingual pretraining,
especially on the cross-source test set?
- **T1** `xlm-roberta-base` – general multilingual model (100 languages)
- **T2** `Davlan/afro-xlmr-base` – XLM-R further pretrained (MLM) on 17 African languages incl. Kinyarwanda
- **T3** `castorini/afriberta_base` – trained from scratch on 11 African languages incl. Kinyarwanda/Kirundi

In [4]:
!python scripts/train_transformer.py --model FacebookAI/xlm-roberta-base --run-name t1_xlmr_base
!python scripts/train_transformer.py --model Davlan/afro-xlmr-base --run-name t2_afroxlmr_base
!python scripts/train_transformer.py --model castorini/afriberta_base --run-name t3_afriberta_base

config.json: 100%|█████████████████████████████| 615/615 [00:00<00:00, 2.95MB/s]
tokenizer_config.json: 100%|█████████████████| 25.0/25.0 [00:00<00:00, 60.2kB/s]
sentencepiece.bpe.model: 100%|██████████████| 5.07M/5.07M [00:00<00:00, 193MB/s]
tokenizer.json: 100%|███████████████████████| 9.10M/9.10M [00:00<00:00, 106MB/s]

model.safetensors: downloading bytes: ██▋                   |  135MB, 8.45MB/s  
model.safetensors: downloading bytes: ████▊                 |  241MB, 18.7MB/s  
model.safetensors: downloading bytes: █████▋                |  287MB, 23.1MB/s  
model.safetensors: downloading bytes: ████████▎             |  425MB, 35.6MB/s  
model.safetensors: downloading bytes: ███████████▏          |  569MB, 48.0MB/s  
model.safetensors: downloading bytes: █████████████▌        |  689MB, 58.0MB/s  
model.safetensors: reconstructing file:  84%|█████ |  938MB / 1.12GB, 73.8MB/s  
model.safetensors: downloading bytes: ██████████████████████|  745MB, 66.2MB/s  
model.safetensors: reconstr

**T4** – context length: the median article is ~300 words (more subword tokens), so 256 tokens truncates many.
Set `BEST` to the best of T1–T3 by **validation** macro-F1 (never pick by test).

In [6]:
import json, glob
for f in sorted(glob.glob("results/t*.json")):
    d = json.load(open(f)); print(f"{d['name']:25s} val mF1 {d['val']['macro_f1']:.4f}  test mF1 {d['test']['macro_f1']:.4f}")

t1_xlmr_base              val mF1 0.4343  test mF1 0.4048
t2_afroxlmr_base          val mF1 0.5294  test mF1 0.4566
t3_afriberta_base         val mF1 0.6053  test mF1 0.4674


In [ ]:
BEST = "castorini/afriberta_base"   # <-- edit after looking at val scores above
!python scripts/train_transformer.py --model $BEST --run-name t4_best_maxlen512 --max-length 512 --batch-size 8 --save-model

In [ ]:
import json, glob
print(sorted(glob.glob("results/*.json")))
for f in sorted(glob.glob("results/e[789]*.json")):
    d = json.load(open(f)); print(f"{d['name']:28s} val mF1 {d['val']['macro_f1']:.4f}  test mF1 {d['test']['macro_f1']:.4f}")


## Download results

In [ ]:
!zip -qr results.zip results
from google.colab import files; files.download("results.zip")

## Publish the final model to the Hugging Face Hub (used by the web app)
Create a **write** token at https://huggingface.co/settings/tokens first.

In [ ]:
from huggingface_hub import login; login()
from transformers import AutoModelForSequenceClassification, AutoTokenizer
path = "models/t4_best_maxlen512/final"
HUB_ID = "Samkwizera/kinnews-topic-classifier"   # <-- your HF username
AutoModelForSequenceClassification.from_pretrained(path).push_to_hub(HUB_ID)
AutoTokenizer.from_pretrained(path).push_to_hub(HUB_ID)